# Бустинги и ансамбли для анализа конверсии

Предсказываем `Revenue` по завершённой сессии Online Shoppers.
Сравниваем библиотеки, подбираем параметры и оцениваем влияние дисбаланса.

## Содержание

1. [Данные](#data)
2. [Бустинги](#boosting)
3. [Optuna](#optuna)
4. [Дисбаланс](#imbalance)
5. [Стекинг](#stacking)
6. [Итоговая оценка](#evaluation)
7. [SHAP](#shap)
8. [Сохранение](#saving)

In [ ]:
import pickle
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import shap
from IPython.display import display, Markdown
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

SEED = 42
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
MODEL_PATH = ROOT / "models/session_advanced_model.pkl"
CAT = ["Month", "OperatingSystems", "Browser", "Region", "TrafficType", "VisitorType", "Weekend"]
optuna.logging.set_verbosity(optuna.logging.WARNING)

<a id="data"></a>
## Данные

Берём исходный CSV без искусственных пропусков. Полные совпадения признаков
остаются в одной группе при разделении и кросс-валидации. Тест – около 20% групп.
Остальные данные используем для выбора моделей на пяти одинаковых фолдах.

In [ ]:
data = pd.read_csv(ROOT / "data/raw/online_shoppers_intention.csv")
X = data.drop(columns="Revenue")
y = data.Revenue.astype(int)
NUM = [c for c in X if c not in CAT]

def folds_for(frame, target):
    groups = pd.factorize(pd.MultiIndex.from_frame(frame))[0]
    return list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(frame, target, groups))

groups = pd.factorize(pd.MultiIndex.from_frame(X))[0]
unique = pd.DataFrame({"group": groups, "target": y})
if unique.groupby("group").target.nunique().max() > 1:
    raise ValueError("Совпадающие признаки имеют разные цели.")
unique = unique.drop_duplicates("group")
train_groups, test_groups = train_test_split(unique.group, stratify=unique.target, test_size=0.2, random_state=SEED)
mask = np.isin(groups, train_groups)
X_train, X_test = X.loc[mask], X.loc[~mask]
y_train, y_test = y.loc[mask], y.loc[~mask]
folds = folds_for(X_train, y_train)
display(pd.DataFrame({"строк": [len(X_train), len(X_test)], "доля покупок": [y_train.mean(), y_test.mean()]}, index=["train", "test"]))
print("Пропусков:", int(X.isna().sum().sum()), "повторов:", int(X.duplicated().sum()))

In [ ]:
def prepare(scale=False):
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale: numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", SkPipeline(numeric_steps), NUM),
        ("cat", SkPipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT),
    ])

def boosting(name, params=None, balance="none", target=None):
    settings = dict(params or {})
    ratio = float((target == 0).sum() / (target == 1).sum()) if balance == "weights" else 1.0
    if name == "XGBoost":
        model = XGBClassifier(random_state=SEED, n_jobs=2, **settings, **({"scale_pos_weight": ratio} if balance == "weights" else {}))
    elif name == "LightGBM":
        model = LGBMClassifier(random_state=SEED, n_jobs=2, verbosity=-1, **settings, **({"scale_pos_weight": ratio} if balance == "weights" else {}))
    else:
        model = CatBoostClassifier(random_seed=SEED, thread_count=2, verbose=False, allow_writing_files=False, **settings, **({"class_weights": [1.0, ratio]} if balance == "weights" else {}))
    return Pipeline([
        ("preprocessor", prepare()),
        ("sampler", RandomUnderSampler(random_state=SEED) if balance == "under" else "passthrough"),
        ("model", model),
    ])

def metrics(target, probability):
    prediction = probability >= 0.5
    return {"ROC-AUC": roc_auc_score(target, probability), "AP": average_precision_score(target, probability),
            "F1": f1_score(target, prediction, zero_division=0), "Precision": precision_score(target, prediction, zero_division=0),
            "Recall": recall_score(target, prediction, zero_division=0)}

all_rows = []
def evaluate_cv(label, factory):
    records = []
    for fit, valid in folds:
        model = factory(X_train.iloc[fit], y_train.iloc[fit])
        started = time.perf_counter()
        model.fit(X_train.iloc[fit], y_train.iloc[fit])
        elapsed = time.perf_counter() - started
        row = metrics(y_train.iloc[valid], model.predict_proba(X_train.iloc[valid])[:, 1])
        row["Fit seconds"] = elapsed
        records.append(row)
    scores = pd.DataFrame(records)
    result = {"Модель": label, **scores.mean().to_dict(), "ROC-AUC std": scores["ROC-AUC"].std(ddof=1)}
    all_rows.append(result)
    print(label, round(result["ROC-AUC"], 5), flush=True)
    return result

<a id="boosting"></a>
## Бустинги

Используем параметры обучения по умолчанию. Фиксируем seed, два потока CPU и отключаем
служебный вывод. Предобработка общая: медиана и one-hot. Деревьям масштабирование не нужно.
Это сравнение библиотек с одинаковой предобработкой, без нативных категорий CatBoost.
Стандартное число деревьев и бюджеты обучения у библиотек различаются.

In [ ]:
baseline_rows = []
for name in ["XGBoost", "LightGBM", "CatBoost"]:
    baseline_rows.append(evaluate_cv(name, lambda frame, target, name=name: boosting(name)))
baselines = pd.DataFrame(baseline_rows).set_index("Модель")
display(baselines.round(4))
best_library = baselines["ROC-AUC"].idxmax()
print("Выбрана библиотека:", best_library)

<a id="optuna"></a>
## Optuna

Проводим 50 полных trials с пятью фолдами внутри objective. Выбираем по среднему ROC-AUC.
Тест не участвует. Диапазоны ограничены, чтобы сравнение можно было повторить на CPU.

In [ ]:
def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
    }
    count = trial.suggest_int("trees", 100, 400, step=50)
    depth = trial.suggest_int("depth", 3, 7)
    if best_library == "CatBoost":
        params.update(iterations=count, depth=depth, l2_leaf_reg=trial.suggest_float("regularization", 1, 20, log=True))
    elif best_library == "LightGBM":
        params.update(n_estimators=count, max_depth=depth, num_leaves=trial.suggest_int("num_leaves", 7, min(63, 2**depth)), reg_lambda=trial.suggest_float("regularization", 0.01, 10, log=True))
    else:
        params.update(n_estimators=count, max_depth=depth, reg_lambda=trial.suggest_float("regularization", 0.01, 10, log=True))
    trial.set_user_attr("model_params", params)
    values = []
    for fit, valid in folds:
        model = boosting(best_library, params)
        model.fit(X_train.iloc[fit], y_train.iloc[fit])
        values.append(roc_auc_score(y_train.iloc[valid], model.predict_proba(X_train.iloc[valid])[:, 1]))
    return float(np.mean(values))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
def progress(study, trial):
    if (trial.number + 1) % 5 == 0: print("Trials:", trial.number + 1, "лучший ROC-AUC:", round(study.best_value, 5), flush=True)
study.optimize(objective, n_trials=50, n_jobs=1, callbacks=[progress])
best_params = study.best_trial.user_attrs["model_params"]
print("Лучшие параметры:", best_params)
print("ROC-AUC:", study.best_value)
plot_optimization_history(study)
plt.tight_layout(); plt.show()
plot_param_importances(study)
plt.tight_layout(); plt.show()
evaluate_cv("Optimized", lambda frame, target: boosting(best_library, best_params))